# Загрузка данных из VitalDB

Этот notebook загружает данные PPG и ABP из VitalDB и сохраняет сырые сегменты для дальнейшей предобработки.

**Параметры данных:**
- Исходная частота дискретизации VitalDB: 500 Hz
- Целевая частота: 125 Hz
- Длина сегмента: 1024 сэмпла (~8.2 секунды)

**Выходные данные:** `data/vitaldb_raw.p` — сырые сегменты без нормализации

## 1. Установка и импорт библиотек

In [1]:
# Установка vitaldb (раскомментировать при первом запуске)
# !pip install vitaldb

In [2]:
import vitaldb
import numpy as np
import pickle
import os
from scipy import signal
from scipy.signal import decimate
from tqdm import tqdm

# Параметры
TARGET_FS = 125          # Целевая частота дискретизации (Hz)
SEGMENT_LENGTH = 1024    # Длина сегмента (samples)
SEGMENT_STEP = 512       # Шаг между сегментами (50% overlap)
VITALDB_FS = 500         # Частота дискретизации VitalDB

print("Libraries imported successfully!")

Libraries imported successfully!


## 2. Поиск случаев с PPG и ABP в VitalDB

In [3]:
# Названия сигналов в VitalDB
PPG_TRACK = 'SNUADC/PLETH'
ABP_TRACK = 'SNUADC/ART'

# Поиск случаев, содержащих оба сигнала
print("Поиск случаев с PPG и ABP...")
cases_with_ppg_abp = vitaldb.find_cases([PPG_TRACK, ABP_TRACK])
print(f"Найдено {len(cases_with_ppg_abp)} случаев с синхронными PPG и ABP")

Поиск случаев с PPG и ABP...
Найдено 3459 случаев с синхронными PPG и ABP


In [4]:
# Просмотр первых 10 случаев
print("Первые 10 случаев:")
for i, case_id in enumerate(cases_with_ppg_abp[:10]):
    print(f"  {i+1}. Case ID: {case_id}")

Первые 10 случаев:
  1. Case ID: 1
  2. Case ID: 3
  3. Case ID: 4
  4. Case ID: 7
  5. Case ID: 10
  6. Case ID: 13
  7. Case ID: 14
  8. Case ID: 16
  9. Case ID: 17
  10. Case ID: 19


## 3. Функции для загрузки и валидации данных

In [5]:
def is_valid_segment(ppg_seg, abp_seg, max_nan_ratio=0.1):
    """
    Проверка валидности сегмента (отсутствие NaN, артефактов).
    
    Args:
        ppg_seg: сегмент PPG
        abp_seg: сегмент ABP
        max_nan_ratio: максимально допустимая доля NaN
    
    Returns:
        True если сегмент валиден
    """
    # Проверка на NaN
    ppg_nan_ratio = np.sum(np.isnan(ppg_seg)) / len(ppg_seg)
    abp_nan_ratio = np.sum(np.isnan(abp_seg)) / len(abp_seg)
    
    if ppg_nan_ratio > max_nan_ratio or abp_nan_ratio > max_nan_ratio:
        return False
    
    # Проверка на плоский сигнал (нет вариации)
    if np.nanstd(ppg_seg) < 1e-6 or np.nanstd(abp_seg) < 1e-6:
        return False
    
    # Проверка на физиологически правдоподобные значения ABP (20-300 mmHg)
    abp_min = np.nanmin(abp_seg)
    abp_max = np.nanmax(abp_seg)
    if abp_min < 20 or abp_max > 300:
        return False
    
    return True


def extract_segments(ppg, abp, segment_length=SEGMENT_LENGTH, step=SEGMENT_STEP):
    """
    Извлечение сегментов из сигналов PPG и ABP.
    
    Args:
        ppg: сигнал PPG
        abp: сигнал ABP
        segment_length: длина сегмента
        step: шаг между сегментами
    
    Returns:
        ppg_segments, abp_segments: списки валидных сегментов
    """
    ppg_segments = []
    abp_segments = []
    
    n_samples = min(len(ppg), len(abp))
    
    for start in range(0, n_samples - segment_length, step):
        end = start + segment_length
        
        ppg_seg = ppg[start:end]
        abp_seg = abp[start:end]
        
        if is_valid_segment(ppg_seg, abp_seg):
            # Интерполяция NaN (если есть небольшое количество)
            if np.any(np.isnan(ppg_seg)):
                ppg_seg = np.interp(
                    np.arange(len(ppg_seg)),
                    np.arange(len(ppg_seg))[~np.isnan(ppg_seg)],
                    ppg_seg[~np.isnan(ppg_seg)]
                )
            if np.any(np.isnan(abp_seg)):
                abp_seg = np.interp(
                    np.arange(len(abp_seg)),
                    np.arange(len(abp_seg))[~np.isnan(abp_seg)],
                    abp_seg[~np.isnan(abp_seg)]
                )
            
            ppg_segments.append(ppg_seg)
            abp_segments.append(abp_seg)
    
    return ppg_segments, abp_segments

## 4. Загрузка и обработка данных

In [6]:
# Параметры загрузки
MAX_CASES = 100              # Максимальное количество случаев для загрузки
MAX_SEGMENTS_PER_CASE = 100 # Максимум сегментов с одного случая

all_ppg_segments = []
all_abp_segments = []

cases_to_process = cases_with_ppg_abp[:MAX_CASES]

print(f"Загрузка данных из {len(cases_to_process)} случаев...")

for case_id in tqdm(cases_to_process, desc="Обработка случаев"):
    try:
        # Загрузка сигналов
        data = vitaldb.load_case(case_id, [PPG_TRACK, ABP_TRACK], 1/VITALDB_FS)
        
        if data is None or len(data) == 0:
            continue
        
        ppg_raw = data[:, 0]  # Первый столбец - PPG
        abp_raw = data[:, 1]  # Второй столбец - ABP
        
        # Ресемплинг к целевой частоте (500 Hz -> 125 Hz)
        ppg_resampled = decimate(ppg_raw, 4, ftype='fir', zero_phase=True)
        abp_resampled = decimate(abp_raw, 4, ftype='fir', zero_phase=True)
        
        # Извлечение сегментов
        ppg_segs, abp_segs = extract_segments(ppg_resampled, abp_resampled)
        
        # Ограничение количества сегментов с одного случая
        if len(ppg_segs) > MAX_SEGMENTS_PER_CASE:
            indices = np.random.choice(len(ppg_segs), MAX_SEGMENTS_PER_CASE, replace=False)
            ppg_segs = [ppg_segs[i] for i in indices]
            abp_segs = [abp_segs[i] for i in indices]
        
        all_ppg_segments.extend(ppg_segs)
        all_abp_segments.extend(abp_segs)
        
    except Exception as e:
        print(f"Ошибка при обработке случая {case_id}: {e}")
        continue

print(f"\nВсего извлечено {len(all_ppg_segments)} валидных сегментов")

Загрузка данных из 100 случаев...


Обработка случаев: 100%|██████████| 100/100 [16:42<00:00, 10.02s/it]



Всего извлечено 9400 валидных сегментов


## 5. Сохранение сырых данных

In [7]:
# Преобразование в numpy массивы
X_raw = np.array(all_ppg_segments)  # (N, 1024)
Y_raw = np.array(all_abp_segments)  # (N, 1024)

print(f"Форма X (PPG): {X_raw.shape}")
print(f"Форма Y (ABP): {Y_raw.shape}")
print(f"\nСтатистика PPG: min={X_raw.min():.3f}, max={X_raw.max():.3f}, mean={X_raw.mean():.3f}")
print(f"Статистика ABP: min={Y_raw.min():.3f}, max={Y_raw.max():.3f}, mean={Y_raw.mean():.3f}")

Форма X (PPG): (9400, 1024)
Форма Y (ABP): (9400, 1024)

Статистика PPG: min=-15.994, max=101.638, mean=38.571
Статистика ABP: min=20.342, max=298.897, mean=83.553


In [8]:
# Создание директории для данных
data_dir = 'raw_data'
os.makedirs(data_dir, exist_ok=True)

# Сохранение сырых данных
vitaldb_raw = {
    'X_raw': X_raw,
    'Y_raw': Y_raw,
    'source': 'VitalDB',
    'fs': TARGET_FS,
    'segment_length': SEGMENT_LENGTH,
    'n_cases': len(cases_to_process),
    'n_segments': len(X_raw)
}

output_path = os.path.join(data_dir, 'vitaldb_raw.p')
with open(output_path, 'wb') as f:
    pickle.dump(vitaldb_raw, f)

print(f"Сырые данные сохранены в {output_path}")
print(f"  - Количество сегментов: {len(X_raw)}")
print(f"  - Размер X_raw: {X_raw.shape}")
print(f"  - Размер Y_raw: {Y_raw.shape}")
print(f"\nДля предобработки используйте _vital_preprocessing.ipynb")

Сырые данные сохранены в raw_data\vitaldb_raw.p
  - Количество сегментов: 9400
  - Размер X_raw: (9400, 1024)
  - Размер Y_raw: (9400, 1024)

Для предобработки используйте _vital_preprocessing.ipynb
